# AI Ritual: RAG in Google Colab — Larger-model path

This notebook is the **GPU-enabled companion** to the Windows-local RAG lab. It uses the same transparent chain—documents, chunks, embeddings, semantic retrieval, and a grounded answer—but uses a larger Hugging Face generative model.

> **Privacy boundary:** Colab and Hugging Face are cloud services. Do not upload confidential, personal, regulated, or client-owned documents unless your organisation has approved this workflow. This notebook is not the “100% local” option; the Windows lab is the offline-after-setup option.

## Before you run

In Colab, select **Runtime → Change runtime type → T4 GPU** (or another available GPU). Free GPU availability is not guaranteed. The notebook checks this requirement before attempting to load the 7B model. The download is free and needs no API key for these public checkpoints.

The main generator is `Qwen/Qwen2.5-7B-Instruct`, loaded in 4-bit NF4 quantization. Hugging Face documents that bitsandbytes reduces memory use through quantized layers, and lists NVIDIA T4-or-newer support for LLM.int8 and Pascal-or-newer for NF4/FP4 quantization. [1]


In [ ]:
# Install the Python packages. This cell may take a few minutes on first use.
!pip -q install --upgrade transformers accelerate bitsandbytes sentence-transformers pypdf


In [ ]:
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError(
        "No GPU detected. In Colab, select Runtime → Change runtime type → T4 GPU, "
        "then run all cells again. The 7B demonstration is intentionally GPU-only."
    )


## 1. Add a small teaching corpus

The cell below creates a fictional restaurant knowledge base. You may instead upload **non-sensitive** `.txt`, `.md`, or text-based `.pdf` files to the `documents` folder. A retrieval system is only as useful as its source material; it cannot answer facts that are absent or poorly represented in those documents.


In [ ]:
DOCUMENT_DIR = Path("documents")
DOCUMENT_DIR.mkdir(exist_ok=True)

sample_text = '''# Guest Experience Playbook

A guest who reports a missing or incorrect item should be thanked, asked for the order reference, and offered either a replacement or a refund for the affected item. A shift manager approves refunds above £25. The team should not promise a delivery time until the delivery partner has confirmed availability.

If a guest reports an allergy-related concern, the shift manager must handle the case immediately. Staff must never assure a guest that a meal is allergen-free unless the kitchen's current allergen matrix confirms it. The incident is recorded in the daily service log.

Reservations for up to six guests can be made online. Groups of seven to sixteen should use the events enquiry form, and the events coordinator responds within two business days. A deposit is required for groups of ten or more. The deposit is refundable if the cancellation is received at least 48 hours before the booking.

Food quality is owned by the head chef. Waiting-time and hospitality comments are owned by the front-of-house manager. Delivery and packaging comments are owned by the operations manager. Price or value comments are reviewed jointly by the general manager and finance lead at the weekly trading meeting.
'''
(DOCUMENT_DIR / "restaurant_playbook.md").write_text(sample_text, encoding="utf-8")
print("Teaching corpus created:", list(DOCUMENT_DIR.iterdir()))

# Optional: upload non-sensitive files manually.
# from google.colab import files
# uploaded = files.upload()
# for filename, data in uploaded.items():
#     (DOCUMENT_DIR / filename).write_bytes(data)


## 2. Read, chunk, and retain source locations

Chunking creates smaller passages that fit alongside a question in the language model’s context. The source name and page/location are carried through the pipeline so that the final answer can be inspected rather than treated as an untraceable output.


In [ ]:
@dataclass
class Chunk:
    source: str
    location: str
    text: str


def chunk_text(source: str, location: str, text: str, size: int = 140, overlap: int = 30) -> List[Chunk]:
    words = text.split()
    step = max(1, size - overlap)
    return [
        Chunk(source, location, " ".join(words[start:start + size]))
        for start in range(0, len(words), step)
        if words[start:start + size]
    ]


def read_documents(folder: Path) -> List[Chunk]:
    from pypdf import PdfReader

    chunks = []
    for path in sorted(folder.iterdir()):
        suffix = path.suffix.lower()
        if suffix in {".txt", ".md"}:
            chunks.extend(chunk_text(path.name, "text file", path.read_text(encoding="utf-8", errors="ignore")))
        elif suffix == ".pdf":
            reader = PdfReader(str(path))
            for page_no, page in enumerate(reader.pages, start=1):
                text = page.extract_text() or ""
                if text.strip():
                    chunks.extend(chunk_text(path.name, f"page {page_no}", text))
    if not chunks:
        raise ValueError("No supported text was found. Add .txt, .md, or text-based .pdf files.")
    return chunks

chunks = read_documents(DOCUMENT_DIR)
print(f"Created {len(chunks)} chunks")
print("\nFirst chunk:\n", chunks[0])


## 3. Build semantic retrieval with a compact embedding model

An embedding model turns text into a vector. By comparing the question vector with chunk vectors, semantic retrieval can match related meanings rather than relying only on exact shared keywords. [2] The embedding model stays small because the larger capacity in this notebook is reserved for **generation**, not retrieval.


In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

chunk_vectors = embedder.encode(
    [chunk.text for chunk in chunks],
    normalize_embeddings=True,
    show_progress_bar=True,
)


def retrieve(question: str, top_k: int = 3) -> List[Tuple[Chunk, float]]:
    question_vector = embedder.encode(question, normalize_embeddings=True)
    scores = np.asarray(chunk_vectors) @ np.asarray(question_vector)
    best = np.argsort(scores)[::-1][:top_k]
    return [(chunks[i], float(scores[i])) for i in best]

question = "How should the restaurant respond to an allergy-related concern?"
evidence = retrieve(question)
for number, (chunk, score) in enumerate(evidence, start=1):
    print(f"[S{number}] {chunk.source} | {chunk.location} | similarity={score:.3f}")
    print(chunk.text[:350], "\n")


## 4. Load a larger instruct model in 4-bit form

Quantization trades some numerical precision for lower memory use. It is an **engineering constraint decision**, not an assurance that two models will give the same quality. The code below makes output more repeatable with low temperature, but it still requires evaluation against known-answer questions.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype="auto",
)
print("Larger instruct model loaded.")


## 5. Generate an evidence-grounded answer

The prompt explicitly limits the model to retrieved sources, requires citations, and defines an abstention phrase. This is a useful control, but it must be tested: poor retrieval, ambiguous sources, or model non-compliance can still produce a wrong answer.


In [ ]:
def make_prompt(question: str, evidence: List[Tuple[Chunk, float]]) -> str:
    source_blocks = []
    for i, (chunk, score) in enumerate(evidence, start=1):
        source_blocks.append(
            f"[S{i}] {chunk.source} ({chunk.location}; relevance={score:.2f})\n{chunk.text}"
        )
    context = "\n\n".join(source_blocks)
    return f'''You are a careful RAG assistant. Answer using ONLY the supplied sources.
If the sources are insufficient, say exactly: "I do not have enough evidence in the supplied documents."
Do not invent policy, figures, or facts. Cite every factual statement with [S1], [S2], etc.

Question: {question}

Sources:
{context}'''


def answer_rag(question: str, top_k: int = 3, max_new_tokens: int = 220) -> Tuple[str, List[Tuple[Chunk, float]]]:
    evidence = retrieve(question, top_k=top_k)
    messages = [
        {"role": "system", "content": "You are precise, concise, and evidence-grounded."},
        {"role": "user", "content": make_prompt(question, evidence)},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), evidence

answer, evidence = answer_rag(question)
print(answer)


## 6. Inspect and evaluate rather than simply admire the answer

Ask a question that the documents **cannot** answer. A good system should abstain rather than make up a policy. Then create a small evaluation sheet with representative questions, expected evidence, and a human judgement of grounding, correctness, and usefulness.


In [ ]:
unknown_question = "Does the restaurant offer a student discount on Fridays?"
unknown_answer, unknown_evidence = answer_rag(unknown_question)
print("Question:", unknown_question)
print("\nRetrieved evidence:")
for i, (chunk, score) in enumerate(unknown_evidence, start=1):
    print(f"[S{i}] {chunk.source} ({chunk.location}); similarity={score:.3f}")
print("\nAnswer:\n", unknown_answer)

# Suggested evaluation table to complete manually:
# | Question | Expected source | Retrieved source? | Factual? | Properly cites? | Abstains when needed? |


## Reflection

The **RAG architecture** has not changed between the Windows and Colab versions. The generator is larger and uses a GPU; the documents still need permission and quality checks, retrieval still determines the evidence, and the answer must still be evaluated. More model capacity can improve helpfulness but cannot replace process design, governance, or a clear decision about where AI is appropriate.

## References

[1]: https://huggingface.co/docs/transformers/quantization/bitsandbytes "Hugging Face — Bitsandbytes quantization"
[2]: https://huggingface.co/learn/llm-course/en/chapter5/6 "Hugging Face LLM Course — Semantic Search with FAISS"
[3]: https://huggingface.co/Qwen/Qwen2.5-7B-Instruct "Qwen — Qwen2.5-7B-Instruct model card"
